In [1]:
import os
import tarfile
import numpy as np
from huggingface_hub import hf_hub_download

# 1. Download the pre-extracted XD-Violence feature tarball from the repository
print("🚚 Downloading pre-extracted XD-Violence feature pack...")
local_tar_path = hf_hub_download(
    repo_id="JunheeLee/RefineVAD_Dataset",
    repo_type="dataset",
    filename="XD_Violence.tar.gz"
)

# 2. Extract the archive into your local workspace folder
extract_path = "./violence_features_multimodal"
if not os.path.exists(extract_path):
    print("📦 Extracting features...")
    with tarfile.open(local_tar_path, "r:gz") as tar:
        tar.extractall(path=extract_path)
    print(f"✅ Extracted completely to: {extract_path}")
else:
    print("ℹ️ Feature folder already extracted.")
# 3. Look inside the directory to see your structural layout
# The files inside are structured exactly into:
#   - visual/ (CLIP ViT-L/14 visual segment embeddings)
#   - textual/ (Aligned clip text embeddings)
print("\n📂 Extracted structures preview:")
print(os.listdir(extract_path))

🚚 Downloading pre-extracted XD-Violence feature pack...
ℹ️ Feature folder already extracted.

📂 Extracted structures preview:
['XD_Violence']


In [51]:
import pandas as pd

visual_dir = os.path.join(extract_path, "XD_Violence", "visual", "ViT14L")
textual_dir = os.path.join(extract_path, "XD_Violence", "textual", "ViT14L")


visual_data = []
visual_label = []
for visual_file in os.listdir(visual_dir):

    label = 0 if 'A' in visual_file.split("_")[-2] else 1
    data_vector = (
        np.load(os.path.join(visual_dir, visual_file))
            .mean(axis=0)
    )
    
    visual_data.append(data_vector)
    visual_label.append(label)

textual_data = []
for textual_file in os.listdir(textual_dir):

    data_vector = (
        np.load(os.path.join(textual_dir, textual_file))
            .mean(axis=0)
    )
    
    textual_data.append(data_vector)

In [52]:
visual_label = np.array(visual_label)
zero_indices = np.where(visual_label == 0)[0]
one_indices = np.where(visual_label == 1)[0]

visual_data = np.array(visual_data)
visual_data = np.concat([visual_data[zero_indices][:500],
                         visual_data[one_indices][:2000]])
visual_label = np.concat([visual_label[zero_indices][:500],
                          visual_label[one_indices][:2000]])

textual_data = np.array(textual_data)
textual_data = np.concat([textual_data[zero_indices][:500],
                          textual_data[one_indices][:2000]])

In [53]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [54]:
x, y = visual_data, visual_label
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [55]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9412 (When flagged positive, accuracy is 94.12%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9408 (When flagged positive, accuracy is 94.08%)
Custom Recall Score:    0.9938 (Captured 99.38% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9485 (When flagged positive, accuracy is 94.85%)
Custom Recall Score:    0.9781 (Captured 97.81% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9381 (When flagged positive, accuracy is 93.81%)
Custom Recall Score:    0.9938 (Captured 99.38% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9238 (When flagged positive, accuracy is 92.38%)
Custom 

In [56]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.93      0.75      0.83       100
           1       0.94      0.98      0.96       400

    accuracy                           0.94       500
   macro avg       0.93      0.87      0.90       500
weighted avg       0.94      0.94      0.94       500

[[ 75  25]
 [  6 394]]


In [57]:
x, y = textual_data, visual_label
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [58]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8020 (When flagged positive, accuracy is 80.20%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8076 (When flagged positive, accuracy is 80.76%)
Custom Recall Score:    0.9969 (Captured 99.69% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8101 (When flagged positive, accuracy is 81.01%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8060 (When flagged positive, accuracy is 80.60%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8128 (When flagged positive, accuracy is 81.28%)
Custo

In [59]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.80      0.08      0.15       100
           1       0.81      0.99      0.89       400

    accuracy                           0.81       500
   macro avg       0.81      0.54      0.52       500
weighted avg       0.81      0.81      0.74       500

[[  8  92]
 [  2 398]]
